# 📝 과제 01: 온라인 쇼핑 상담 LoRA 어댑터 학습과 에폭별 어댑터 저장

## 만들 결과

교안 01의 LoRA 학습을 지난 시간 과제의 온라인 쇼핑 상담(전자상거래법) 데이터에 같은 순서로 적용합니다.

**배우는 것** (문항 순서)
1. `LoraConfig`, `get_peft_model`로 어댑터를 붙이고 학습 파라미터 수와 처음 값($B$는 0)을 확인합니다.
2. `SFTTrainer`로 4에폭 학습하며 에폭마다 어댑터를 저장합니다.
3. 학습 전·후 답을 비교하고, 어댑터를 끈 답과 다시 불러온 어댑터의 답을 확인합니다.

**만들 것**
- **학습:** 체크포인트 `output/commerce_lora_lr2e-4/checkpoint-<갱신 횟수>` 네 개(과제 02가 평가), `eval_log`, `check_table`의 `"LoRA 답변"` 열, `adapter_off_answers`
- **저장:** `checkpoint_table`, 어댑터 폴더 `output/commerce_lora_adapter`, 다시 불러온 `reloaded_model`

## 준비: 데이터와 제공 코드

- **파일:** 학습 `data/과제/commerce_train.jsonl`, 검증 `data/과제/commerce_val.jsonl`, 일반 질문 `data/본실습/general_questions.jsonl`(정답 답변 키 `reference_answer`). 범위 밖 질문 행의 `answer`는 거절 답 `REFUSAL_ANSWER_COMMERCE`입니다.
- **제공 코드:** 지난 시간 과제와 같은 준비(`output_dir`, `device`, `MODEL_NAME`, `tokenizer`, MLflow 실험 `commerce-lora`), 함수 `generate_reply(model, question, system_prompt=SYSTEM_PROMPT_COMMERCE)`, 데이터 `train_dataset`, `val_dataset`, 학습 전 모델 `base_model`, 학습 전 답 테이블 `check_table`
- **교안 01과 다른 점:** 학습 데이터가 135행이라 1에폭이 34번 갱신입니다. `logging_steps`는 10, `max_length`는 256이고 나머지 설정은 교안 01과 같습니다.
- **작성:** 요구사항 번호와 같은 `TODO (번호)` 아래의 `...`를 채우고, 바로 아래 `[자가채점]` 셀을 실행합니다.
- **순서:** 앞 문항의 변수를 쓰므로 처음부터 차례로 풉니다.
- **실행 환경:** 학습이 있으므로 교안과 같은 RunPod GPU에서 실행합니다.

In [ ]:
import json
import platform
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

material_dir = Path(".")  # data가 있는 단원 폴더. 정답 노트북에서는 ".."
data_dir = material_dir / "data"
output_dir = material_dir / "output"
output_dir.mkdir(exist_ok=True)

# 계산 장치: RunPod처럼 GPU(cuda)가 있으면 GPU, 없으면 CPU에서 계산합니다
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("계산 장치:", device, "|", torch.cuda.get_device_name(0))
else:
    device = torch.device("cpu")
    print("계산 장치:", device)

# 테이블의 긴 답변이 잘리지 않게 칸의 글자 수 제한을 풉니다
pd.set_option("display.max_colwidth", None)

# 그래프 글꼴: 영문, 숫자, 기호는 DejaVu Sans, 한글은 운영체제의 한글 글꼴이 맡습니다
from matplotlib import font_manager

if platform.system() == "Windows":
    korean_font = "Malgun Gothic"
elif platform.system() == "Darwin":
    korean_font = "AppleGothic"
else:
    korean_font = "NanumGothic"
    # 리눅스(RunPod): 글꼴을 설치하기 전에 만든 matplotlib 글꼴 목록에는 나눔고딕이 없으므로, 설치된 파일을 직접 등록합니다
    for font_path in font_manager.findSystemFonts():
        if "NanumGothic" in font_path:
            font_manager.fontManager.addfont(font_path)
plt.rcParams["font.family"] = ["DejaVu Sans", korean_font]
plt.rcParams["axes.unicode_minus"] = False  # 음수 부호가 네모로 깨지지 않게 합니다

In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-0.8B"

# 토크나이저: 학습 전후 모델이 같은 토크나이저를 씁니다
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("대화 끝 토큰:", tokenizer.eos_token, "| 패딩 토큰:", tokenizer.pad_token)

In [ ]:
# 지난 시간 과제 데이터의 시스템 프롬프트와 범위 밖 거절 답입니다(학습 데이터의 messages와 글자까지 같습니다)
SYSTEM_PROMPT_COMMERCE = "너는 온라인 쇼핑 상담 챗봇이다. 전자상거래법 조문에 근거해 존댓말로 간결히 답한다."
REFUSAL_ANSWER_COMMERCE = "이 상담 챗봇이 다루는 전자상거래법 조문으로는 답할 수 없는 질문입니다. 이 내용은 담당 기관이나 전문가에게 문의해 주세요."

# 대화 끝 토큰 <|im_end|>의 id. 생성이 이 토큰에서 멈추도록 eos_token_id로 넘깁니다
END_OF_TURN_ID = tokenizer.convert_tokens_to_ids("<|im_end|>")


def generate_reply(model, question, system_prompt=SYSTEM_PROMPT_COMMERCE, max_new_tokens=256):
    """질문 하나를 학습 데이터와 같은 chat template으로 감싸 모델이 새로 만든 답만 돌려줍니다.

    system_prompt=None 이면 시스템 메시지 없이 묻습니다(일반 능력 점검용).
    """
    messages = []
    if system_prompt is not None:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": question})
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,  # 끝에 assistant 머리를 붙여 모델이 답을 이어 쓰게 합니다
        enable_thinking=False,       # 빈 생각 블록(<think></think>)을 붙여 생각 과정(thinking) 없이 바로 답합니다
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,         # 새로 만들 토큰 수의 상한. 답이 이보다 길면 끝까지 쓰기 전에 멈춥니다
            do_sample=False,                       # 매번 확률이 가장 높은 토큰을 골라(그리디 생성) 같은 질문에 같은 답이 나옵니다
            eos_token_id=END_OF_TURN_ID,           # 대화 끝 토큰 <|im_end|>에서 멈춥니다
            pad_token_id=tokenizer.pad_token_id,   # 넘기지 않으면 패딩 토큰 경고가 나옵니다
        )

    # 입력 프롬프트 길이만큼 잘라 새로 만든 토큰만 글자로 바꿉니다
    new_ids = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True)


def load_jsonl(path):
    """JSONL 파일을 한 줄에 딕셔너리 하나씩 읽어 리스트로 돌려줍니다(지난 시간과 같은 함수)."""
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def to_prompt_completion(row):
    """messages를 앞부분(system이 있으면 system, user)과 배울 답(마지막 assistant)으로 나눕니다."""
    prompt = row["messages"][:-1]      # 마지막 답 앞의 메시지(system, user)
    completion = row["messages"][-1:]  # 마지막 assistant 메시지 하나
    return {"prompt": prompt, "completion": completion}

In [ ]:
import mlflow

# 실험 기록은 output/mlflow.db 한 파일에 남깁니다(별도 서버 없이 sqlite)
mlflow.set_tracking_uri(f"sqlite:///{output_dir / 'mlflow.db'}")
mlflow.set_experiment("commerce-lora")
print("MLflow 기록 위치:", mlflow.get_tracking_uri())

In [ ]:
from datasets import Dataset

# 학습, 검증 JSONL 을 읽습니다(한 줄이 question, answer, messages 를 가진 딕셔너리 하나)
train_rows = load_jsonl(data_dir / "과제" / "commerce_train.jsonl")
val_rows = load_jsonl(data_dir / "과제" / "commerce_val.jsonl")

# 행마다 messages 를 prompt(시스템, 질문)와 completion(배울 답)으로 나눠 SFTTrainer 가 읽는 Dataset 으로 만듭니다
train_dataset = Dataset.from_list([to_prompt_completion(row) for row in train_rows])
val_dataset = Dataset.from_list([to_prompt_completion(row) for row in val_rows])
print("학습 행:", len(train_dataset), "| 검증 행:", len(val_dataset))

### 학습 전 모델과 학습 전 답

교안 01과 같은 코드라 실행만 합니다.

- **먼저 모으는 이유:** `get_peft_model`이 `base_model`의 층을 그 자리에서 바꾸므로, 학습 전 답은 어댑터를 붙이기 전에 모읍니다.
- **질문 여섯 개:** 학습 데이터 2, 검증 데이터 1, 범위 밖 질문 1, 일반 질문 2(시스템 프롬프트 없이)
- **`check_table`:** 학습한 모델의 답은 학습 뒤 이 테이블에 열로 더합니다.

In [ ]:
# 학습 전 모델(Base): 지난 시간과 같은 float32(파라미터 하나에 4바이트)
# 이 모델에 어댑터를 붙이면 층이 그 자리에서 바뀌므로, 학습 전 답은 다음 셀에서 먼저 모읍니다
base_model =AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=torch.float32).to(device)

In [ ]:
# 학습 데이터에서 정답 답변이 거절 답인 첫 행(범위 밖 질문을 거절하도록 배웠는지 볼 행)
for row in train_rows:
    if row["answer"] == REFUSAL_ANSWER_COMMERCE:
        refusal_row = row
        break

# 쇼핑과 상관없는 상식 질문(정답 답변은 reference_answer). 학습 뒤에도 원래 상식이 남았는지 봅니다
general_rows = load_jsonl(data_dir / "본실습" / "general_questions.jsonl")

# 학습 전, 후에 똑같이 물을 질문 여섯 개와 그 종류(두 리스트는 같은 순서로 짝을 이룹니다)
check_rows = [train_rows[0], train_rows[90], val_rows[0], refusal_row, general_rows[0], general_rows[12]]
check_kinds = ["학습 데이터", "학습 데이터", "검증 데이터", "범위 밖 질문", "일반 질문", "일반 질문"]


def answer_check_questions(model):
    """check_rows의 질문마다 model이 한 답을 같은 순서의 리스트로 돌려줍니다. 일반 질문은 시스템 프롬프트 없이 묻습니다."""
    answers = []
    for row, kind in zip(check_rows, check_kinds):
        # 일반 질문은 상담 챗봇 역할(시스템 프롬프트) 없이, 나머지는 기본 시스템 프롬프트로 묻습니다
        if kind == "일반 질문":
            answers.append(generate_reply(model, row["question"], system_prompt=None))
        else:
            answers.append(generate_reply(model, row["question"]))
    return answers


# 정답 답변: 앞 네 행(쇼핑 상담 학습, 검증 데이터)은 answer, 뒤 두 행(일반 질문 파일)은 reference_answer 키에 있습니다
reference_answers = [row["answer"] for row in check_rows[:4]] + [row["reference_answer"] for row in check_rows[4:]]

# 종류, 질문, 정답 답변, Base 답을 한 테이블에 둡니다. 학습 뒤 "LoRA 답변" 열을 여기에 더합니다
check_table = pd.DataFrame({
    "종류": check_kinds,
    "질문": [row["question"] for row in check_rows],
    "정답 답변": reference_answers,
    "Base 답변": answer_check_questions(base_model),
})
display(check_table)

## 1. 어댑터 설정과 붙이기: LoraConfig, get_peft_model

**목적:** 어댑터를 붙인 뒤 학습하는 값이 전체의 1% 미만이고 베이스 가중치는 고정되는지 확인합니다.

**요구사항**
- (1) `LoraConfig`에 `r=8`, `lora_alpha=16`, `target_modules="all-linear"`, `lora_dropout=0.05`, `task_type="CAUSAL_LM"`을 넣어 **`lora_config`** 를 만듭니다.
- (2) `get_peft_model(base_model, lora_config)`로 **`lora_model`** 을 만듭니다.
- (3) `lora_model.get_nb_trainable_parameters()`의 두 반환값(학습 파라미터 수, 전체 파라미터 수)을 **`trainable_count`**, **`all_count`** 에 담습니다.
- (4) 첫 디코더 층의 MLP `gate_proj`인 `lora_model.base_model.model.model.layers[0].mlp.gate_proj`를 **`gate_layer`** 에 담습니다. 학습 전이므로 $B$는 모두 0입니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 3-2 LoraConfig 셀, 4-1 get_peft_model 셀, 4-2 gate_proj 를 꺼내는 셀과 같은 방법입니다.
```

</details>

In [ ]:
from peft import LoraConfig, get_peft_model

# TODO (1): LoraConfig 로 설정을 만들어 lora_config 에 담으세요
lora_config = ...

# TODO (2): get_peft_model 로 어댑터를 붙여 lora_model 에 담으세요
lora_model = ...

# TODO (3): 학습 파라미터 수와 전체 파라미터 수를 담으세요
trainable_count, all_count = ...

# 학습 파라미터 수, 전체 파라미터 수, 비율(%)을 한 줄로 찍습니다
lora_model.print_trainable_parameters()

In [ ]:
# TODO (4): 첫 디코더 층의 mlp.gate_proj 를 gate_layer 에 담으세요
gate_layer = ...

# 출력에서 base_layer(원래 Linear)와 lora_A, lora_B 가 한 층 안에 함께 있는지 봅니다
display(gate_layer)

In [ ]:
# [자가채점] 설정 값, 학습 파라미터 수, gate_proj 의 A, B 를 확인합니다
from peft import PeftModel

# 설정 값
assert isinstance(lora_config, LoraConfig), "lora_config는 peft의 LoraConfig입니다. `...`를 채웠는지 확인하세요."
assert lora_config.r == 8 and lora_config.lora_alpha == 16 and lora_config.lora_dropout == 0.05, "r=8, lora_alpha=16, lora_dropout=0.05를 넣으세요."
assert lora_config.task_type == "CAUSAL_LM", "task_type=\"CAUSAL_LM\"을 넣으세요."

# 어댑터를 붙인 모델, 학습 파라미터 수(전체의 1% 미만), 어댑터를 붙인 층 수
assert isinstance(lora_model, PeftModel), "lora_model은 get_peft_model(base_model, lora_config)이 돌려준 모델입니다."
assert (trainable_count, all_count) == lora_model.get_nb_trainable_parameters(), "trainable_count, all_count 는 lora_model.get_nb_trainable_parameters()의 두 반환값입니다."
assert 0 < trainable_count < all_count * 0.01, "학습 파라미터는 0보다 많고 전체의 1% 미만이어야 합니다. target_modules=\"all-linear\"와 r 값을 확인하세요."
assert len(lora_model.base_model.targeted_module_names) == 186, f"어댑터를 붙인 층이 {len(lora_model.base_model.targeted_module_names)}개입니다(기대값 186개). target_modules=\"all-linear\"를 확인하세요."

# gate_proj 의 A(r x 입력 1024), B(출력 3584 x r) 모양, 학습 전 B 는 0, 베이스 고정과 A, B 학습
assert tuple(gate_layer.lora_A["default"].weight.shape) == (8, 1024) and tuple(gate_layer.lora_B["default"].weight.shape) == (3584, 8), f"A 모양 {tuple(gate_layer.lora_A['default'].weight.shape)}, B 모양 {tuple(gate_layer.lora_B['default'].weight.shape)}입니다. gate_layer 가 첫 층의 mlp.gate_proj 인지 확인하세요."
assert gate_layer is lora_model.base_model.model.model.layers[0].mlp.gate_proj, "gate_layer가 첫 디코더 층의 mlp.gate_proj 가 아닙니다. 다른 층을 담지 않았는지 확인하세요."
assert gate_layer.lora_B["default"].weight.abs().max().item() == 0, "학습 전 B는 모두 0이어야 합니다. 이미 학습했다면 학습 전 모델 셀부터 다시 실행하세요."
assert gate_layer.base_layer.weight.requires_grad is False and gate_layer.lora_A["default"].weight.requires_grad is True, "베이스 가중치는 고정(requires_grad False), A, B는 학습합니다."
print("✅ 통과!")

### 어댑터를 붙인 층과 어댑터 파라미터(제공 코드)

**목적:** 어댑터를 붙인 층을 이름 끝부분별 `linear_table`과 층 종류별 `group_table`로 모아 층 수와 어댑터 파라미터 합을 테이블로 만듭니다. 실행만 합니다.

In [ ]:
# 제공 코드: get_peft_model 이 어댑터를 붙인 층을 이름 끝부분(예: gate_proj)별로 모읍니다
# targeted_module_names: 어댑터를 붙인 층의 전체 이름 목록(예: model.layers.0.mlp.gate_proj)
targeted_names = lora_model.base_model.targeted_module_names
linear_counts = {}  # 끝부분 이름 -> 층 수
linear_params = {}  # 끝부분 이름 -> 어댑터 파라미터 수 합
for name in targeted_names:
    # 전체 이름으로 그 LoRA 층 객체를 꺼냅니다
    layer = lora_model.base_model.model.get_submodule(name)

    # 이름 예: model.layers.0.mlp.gate_proj -> 끝부분 gate_proj
    short_name = name.split(".")[-1]
    if short_name not in linear_counts:
        linear_counts[short_name] = 0
        linear_params[short_name] = 0

    # 층 하나의 어댑터 파라미터 = A 원소 수(r x 입력 크기) + B 원소 수(출력 크기 x r)
    linear_counts[short_name] += 1
    linear_params[short_name] += layer.lora_A["default"].weight.numel() + layer.lora_B["default"].weight.numel()

# 이름 끝부분마다 한 행
linear_rows = []
for short_name in linear_counts:
    linear_rows.append({"층 이름": short_name, "층 수": linear_counts[short_name], "어댑터 파라미터 합": linear_params[short_name]})
linear_table = pd.DataFrame(linear_rows)
display(linear_table)

In [ ]:
# 제공 코드: 층 종류마다 층 수와 어댑터 파라미터 수를 더합니다
# 층 종류 -> 그 종류에 속한 이름 끝부분(교안 01 대상 층 그림과 같은 묶음)
layer_groups = {
    "선형 어텐션": ["in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj"],
    "전체 어텐션": ["q_proj", "k_proj", "v_proj", "o_proj"],
    "MLP": ["gate_proj", "up_proj", "down_proj"],
}
group_rows = []
for group_name in layer_groups:
    # linear_table 에서 이 종류의 이름인 행만 골라(isin) 두 열을 더합니다
    rows_in_group = linear_table[linear_table["층 이름"].isin(layer_groups[group_name])]
    group_rows.append({"층 종류": group_name, "층 수": int(rows_in_group["층 수"].sum()),
                       "어댑터 파라미터 합": int(rows_in_group["어댑터 파라미터 합"].sum())})

# 층 종류를 행 이름으로 둡니다
group_table =pd.DataFrame(group_rows).set_index("층 종류")
display(group_table)

## 2. 어댑터 학습과 어댑터 끄기: SFTTrainer, disable_adapter

**목적:** 에폭마다 어댑터를 저장하며 학습하고(과제 02가 이 체크포인트로 에폭을 고릅니다), 어댑터를 끄면 학습 전 답으로 돌아가는지 확인합니다.

**요구사항**
- (1) `SFTConfig`에 아래 값을 넣어 **`sft_config`** 를 만듭니다.

```text
output_dir                   str(output_dir / "commerce_lora_lr2e-4")
learning_rate                2e-4
num_train_epochs             4
per_device_train_batch_size  4
seed                         0
gradient_checkpointing       False
max_length                   256
eval_strategy                "epoch"
save_strategy                "epoch"
save_only_model              True
logging_steps                10
eval_on_start                True
report_to                    ["mlflow"]
run_name                     "lora_lr_2e-4"
```

- (2) `SFTTrainer`에 `model=lora_model`, `args=sft_config`, `train_dataset=train_dataset`, `eval_dataset=val_dataset`, `processing_class=tokenizer`를 넘겨 **`trainer`** 를 만듭니다.
- (3) 셀의 MLflow run 안에서 `trainer.train()`의 반환값을 **`train_output`** 에 담습니다. 끝까지 학습하면 136번(에폭당 34번 × 4에폭) 갱신합니다.
- (4) `trainer.state.log_history`를 `pd.DataFrame`으로 바꿔 **`log_table`** 에 담고, `"eval_loss"`가 있는 행만 남겨 **`eval_log`** 에 담습니다(학습 전과 에폭 1~4, 다섯 행).
- (5) `lora_model.eval()`로 드롭아웃을 끈 뒤, `answer_check_questions(lora_model)`의 답을 `check_table`의 새 열 `"LoRA 답변"`에 담습니다.
- (6) `with lora_model.disable_adapter():` 블록 안에서 `answer_check_questions(lora_model)`의 답을 **`adapter_off_answers`** 에 담습니다. 어댑터는 블록 안에서만 꺼집니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 5-2부터 5-5까지(학습, 학습 곡선, 학습 전·후 답, 어댑터 끄기) 셀을 차례로 나눠 담은 모양입니다.
eval_loss 가 있는 행만 남길 때는 그 열이 빈 행을 빼는 DataFrame 메서드를 씁니다.
```

</details>

In [ ]:
from trl import SFTConfig, SFTTrainer

# TODO (1): 위 값을 키워드 인자로 넣어 sft_config 를 만드세요
sft_config = SFTConfig(
    ...
)

# TODO (2): 어댑터를 붙인 모델로 학습기를 만드세요
trainer = ...

# MLflow run 하나에 데이터와 LoRA 설정을 기록합니다. 손실은 report_to 설정으로 학습기가 같은 run 에 남깁니다
with mlflow.start_run(run_name=sft_config.run_name):
    mlflow.log_params({"train_file": "commerce_train.jsonl", "train_rows": len(train_dataset),
                       "val_file": "commerce_val.jsonl", "val_rows": len(val_dataset),
                       "lora_r": lora_config.r, "lora_alpha": lora_config.lora_alpha, "target_modules": "all-linear"})
    # TODO (3): 학습하고 반환값을 train_output 에 담으세요
    train_output = ...

In [ ]:
# TODO (4): 학습 기록을 테이블로 바꾸고 eval_loss 가 있는 행만 남기세요
log_table = ...
eval_log = ...

# step: 그때까지 갱신 횟수, epoch: 에폭, eval_loss: 검증 손실
display(eval_log[["step", "epoch", "eval_loss"]].round(4))

In [ ]:
# TODO (5): 평가 모드로 바꾸고 "LoRA 답변" 열을 더하세요
...
check_table["LoRA 답변"] = ...
display(check_table)

In [ ]:
# TODO (6): disable_adapter 블록 안에서 답을 adapter_off_answers 에 담으세요
...

# 베이스 가중치가 그대로라면 어댑터를 끈 답은 학습 전 Base 답과 같아야 합니다
print("어댑터를 끈 답이 Base 답과 여섯 개 모두 같은가:", adapter_off_answers == list(check_table["Base 답변"]))

In [ ]:
# [자가채점] 설정, 학습, eval_log, 체크포인트, LoRA 답변, 어댑터를 끈 답을 확인합니다
# 학습 설정 값
assert isinstance(sft_config, SFTConfig), "sft_config는 trl의 SFTConfig입니다. `...`를 채웠는지 확인하세요."
assert abs(sft_config.learning_rate - 2e-4) < 1e-12, f"learning_rate가 {sft_config.learning_rate}입니다. 2e-4를 넣으세요."
assert sft_config.num_train_epochs == 4 and sft_config.max_steps == -1 and sft_config.per_device_train_batch_size == 4, "num_train_epochs=4, per_device_train_batch_size=4를 넣고 max_steps는 넣지 마세요."
assert sft_config.max_length == 256 and sft_config.seed == 0 and sft_config.gradient_checkpointing is False, "max_length=256, seed=0, gradient_checkpointing=False를 넣으세요."
assert sft_config.eval_strategy == "epoch" and sft_config.save_strategy == "epoch" and sft_config.save_only_model is True, "eval_strategy=\"epoch\", save_strategy=\"epoch\", save_only_model=True를 넣으세요(에폭마다 어댑터를 저장합니다)."
assert sft_config.logging_steps == 10 and sft_config.eval_on_start is True, "logging_steps=10, eval_on_start=True를 넣으세요."
assert "mlflow" in sft_config.report_to and sft_config.run_name == "lora_lr_2e-4", "report_to=[\"mlflow\"], run_name=\"lora_lr_2e-4\"를 넣으세요."
assert sft_config.output_dir.endswith("commerce_lora_lr2e-4"), f"output_dir가 {sft_config.output_dir}입니다. str(output_dir / \"commerce_lora_lr2e-4\")를 넣으세요."

# 학습기, 끝까지 돈 갱신 횟수
assert isinstance(trainer, SFTTrainer) and trainer.args is sft_config, "trainer는 sft_config를 args로 넘긴 SFTTrainer입니다."
assert trainer.eval_dataset is not None and len(trainer.eval_dataset) == len(val_dataset), "trainer의 검증 데이터 행 수가 val_dataset과 다릅니다. eval_dataset=val_dataset을 넘겼는지 확인하세요."
assert train_output.global_step == 136, f"global_step이 {train_output.global_step}입니다(기대값 136). trainer.train()이 끝까지 돌았는지 확인하세요."

# 학습 전과 에폭마다 한 행인 eval_log, 에폭마다 저장한 체크포인트
assert isinstance(eval_log, pd.DataFrame) and "eval_loss" in eval_log.columns and eval_log["eval_loss"].notna().all(), "eval_log는 log_table 에서 eval_loss 가 있는 행만 남긴 테이블입니다."
assert [round(epoch) for epoch in eval_log["epoch"]] == [0, 1, 2, 3, 4], f"eval_log의 epoch이 {list(eval_log['epoch'])}입니다. eval_on_start=True, eval_strategy=\"epoch\"를 확인하세요."
for step in [34, 68, 102, 136]:
    assert (output_dir / "commerce_lora_lr2e-4" / f"checkpoint-{step}" / "adapter_model.safetensors").exists(), f"checkpoint-{step} 에 어댑터가 없습니다. save_strategy=\"epoch\"를 확인하세요."

# check_table 에 더한 "LoRA 답변" 열
assert list(check_table.columns) == ["종류", "질문", "정답 답변", "Base 답변", "LoRA 답변"], f"check_table의 열이 {list(check_table.columns)}입니다. 준비에서 만든 테이블에 \"LoRA 답변\" 열 하나를 더하세요."
for answer in check_table["LoRA 답변"]:
    assert isinstance(answer, str) and answer.strip() != "", "LoRA 답변이 비어 있습니다. answer_check_questions(lora_model)의 반환값을 담으세요."
assert list(check_table["LoRA 답변"]) != list(check_table["Base 답변"]), "LoRA 답변이 Base 답변과 여섯 개 모두 같습니다. 학습한 lora_model 에 다시 물었는지 확인하세요."

# 어댑터를 끈 답은 Base 답과 같고, 블록을 나온 뒤에는 어댑터가 다시 켜졌는지
assert isinstance(adapter_off_answers, list) and len(adapter_off_answers) == len(check_rows), "adapter_off_answers는 질문 여섯 개의 답 리스트입니다."
assert adapter_off_answers == list(check_table["Base 답변"]), "어댑터를 끈 답이 Base 답과 다릅니다. disable_adapter 블록 안에서 lora_model 에 물었는지 확인하세요."
assert gate_layer.disable_adapters is False, "블록을 나온 뒤에도 어댑터가 꺼져 있습니다. with 블록 안에서만 끄세요."
print("✅ 통과!")

## 3. 어댑터 저장과 다시 불러오기: save_pretrained, PeftModel.from_pretrained

**목적:** 학습한 어댑터가 베이스 모델 없이 작은 파일로 저장되고, 새 베이스 모델에 다시 불러 같은 답을 내는지 확인합니다.

**요구사항**
- (1) `output/commerce_lora_lr2e-4`의 체크포인트마다 한 행인 **`checkpoint_table`** 을 만듭니다. 열은 이 순서입니다.
  - `"에폭"`: 1~4
  - `"체크포인트"`: 폴더 이름 `checkpoint-<에폭 * steps_per_epoch>`
  - `"어댑터 파일(MB)"`: 폴더 안 `adapter_model.safetensors` 크기를 `1024 ** 2`로 나눠 소수 둘째 자리로 반올림
- (2) 셀이 읽어 둔 `adapter_config`(마지막 체크포인트의 `adapter_config.json`)에서 `"base_model_name_or_path"`, `"r"`, `"lora_alpha"`, `"target_modules"`만 이 순서로 남겨 **`adapter_info`** 에 담습니다.
- (3) `lora_model.save_pretrained`로 `output_dir / "commerce_lora_adapter"`에 어댑터를 저장합니다.
- (4) 그 폴더의 파일 이름을 정렬한 리스트 **`saved_files`** 를 만듭니다.
- (5) 학습 전 모델을 준비 셀처럼(`dtype=torch.float32`, `.to(device)`) 새로 불러 **`reload_base`** 에 담습니다.
- (6) `PeftModel.from_pretrained(reload_base, output_dir / "commerce_lora_adapter")`로 **`reloaded_model`** 을 만듭니다.
- (7) `check_rows`의 첫 질문을 `generate_reply`(시스템 프롬프트는 기본값)로 `reloaded_model`에 물어 답을 **`reloaded_answer`** 에 담습니다.

<details><summary>힌트</summary>

```text
접근방법: 교안 01 6-1부터 6-3까지 셀과 같은 방법입니다. 교안은 체크포인트 하나만 보지만 여기서는 에폭 반복문 안에서 체크포인트 폴더마다 경로를 만듭니다.
파일 크기는 경로의 stat 결과에서, 폴더 안 파일은 경로의 iterdir 로 얻습니다.
```

</details>

In [ ]:
import math

# 한 에폭의 갱신 횟수: 학습 행 수를 배치 크기로 나눠 올림합니다(마지막 덜 찬 배치도 한 번 갱신)
steps_per_epoch = math.ceil(len(train_dataset) / 4)
checkpoint_rows = []
for epoch in range(1, 4 + 1):
    # TODO (1): 체크포인트 폴더의 "에폭", "체크포인트", "어댑터 파일(MB)" 딕셔너리를 checkpoint_rows 에 더하세요
    ...

# 딕셔너리 하나가 한 행이 됩니다
checkpoint_table = pd.DataFrame(checkpoint_rows)
display(checkpoint_table)

In [ ]:
# adapter_config.json: 이 어댑터를 어느 베이스 모델의 어느 층에 어떤 r 로 얹는지 적은 설정 파일
with open(output_dir / "commerce_lora_lr2e-4" / "checkpoint-136" / "adapter_config.json", encoding="utf-8") as f:
    adapter_config = json.load(f)

# TODO (2): 네 키만 순서대로 남긴 adapter_info 를 만드세요
adapter_info = ...
display(adapter_info)

In [ ]:
# TODO (3): save_pretrained 로 어댑터를 저장하세요
...

# TODO (4): 폴더 안 파일 이름을 saved_files 에 모으세요
# iterdir 는 폴더 안 경로를 하나씩 돌려줍니다
saved_files = []
for path in (output_dir / "commerce_lora_adapter").iterdir():
    ...
saved_files.sort()
print("저장한 파일:", saved_files)

In [ ]:
from peft import PeftModel

# TODO (5): 학습 전 모델을 새로 불러 reload_base 에 담으세요
reload_base = ...

# TODO (6): 저장한 어댑터를 reload_base 에 얹어 reloaded_model 에 담으세요
reloaded_model = ...

# TODO (7): check_rows 의 첫 질문을 reloaded_model 에 물으세요
reloaded_answer = ...

# 같은 베이스, 같은 어댑터 값이면 그리디 생성 답이 저장 전 LoRA 답과 글자까지 같습니다
print("다시 불러온 어댑터의 답이 저장 전 답과 같은가:", reloaded_answer == check_table["LoRA 답변"][0])

In [ ]:
# [자가채점] 체크포인트 테이블, 파일 크기, 설정 파일, 저장한 파일, 다시 불러온 어댑터를 확인합니다
# 체크포인트 테이블의 열 순서, 에폭 번호, 폴더 이름
assert isinstance(checkpoint_table, pd.DataFrame) and list(checkpoint_table.columns) == ["에폭", "체크포인트", "어댑터 파일(MB)"], "checkpoint_table의 열은 \"에폭\", \"체크포인트\", \"어댑터 파일(MB)\" 순서입니다."
assert list(checkpoint_table["에폭"]) == [1, 2, 3, 4], f"\"에폭\" 열이 {list(checkpoint_table['에폭'])}입니다. 갱신 횟수가 아니라 에폭 번호 1~4를 담으세요."
assert list(checkpoint_table["체크포인트"]) == ['checkpoint-34', 'checkpoint-68', 'checkpoint-102', 'checkpoint-136'], f"체크포인트 이름이 {list(checkpoint_table['체크포인트'])}입니다. 이름은 checkpoint-<에폭 * steps_per_epoch> 입니다."

# 크기 열이 실제 파일 크기와 같고, 학습 파라미터 수 x 4바이트(float32)와 맞는지
for row in checkpoint_table.to_dict("records"):
    file_mb = round((output_dir / "commerce_lora_lr2e-4" / row["체크포인트"] / "adapter_model.safetensors").stat().st_size / 1024 ** 2, 2)
    assert row["어댑터 파일(MB)"] == file_mb, f"{row['체크포인트']} 행의 크기 {row['어댑터 파일(MB)']}MB가 그 폴더 파일({file_mb}MB)과 다릅니다. 1024 ** 2 로 나눠 소수 둘째 자리로 반올림하세요."
expected_mb = trainable_count * 4 / 1024 ** 2
for size_mb in checkpoint_table["어댑터 파일(MB)"]:
    assert abs(size_mb - expected_mb) < expected_mb * 0.01, f"어댑터 파일 {size_mb}MB가 학습 파라미터 수 * 4바이트(약 {expected_mb:.2f}MB)와 다릅니다. 크기를 1024 ** 2 로 나눴는지 확인하세요."

# adapter_info 의 네 키와 값
assert isinstance(adapter_info, dict) and list(adapter_info) == ["base_model_name_or_path", "r", "lora_alpha", "target_modules"], f"adapter_info의 키가 {list(adapter_info)}입니다. 지문의 네 키만 순서대로 남기세요."
assert adapter_info["base_model_name_or_path"] == MODEL_NAME and adapter_info["r"] == 8 and adapter_info["lora_alpha"] == 16, "adapter_info의 베이스 모델, r, lora_alpha를 확인하세요."
assert sorted(adapter_info["target_modules"]) == sorted(linear_table["층 이름"]), f"저장된 target_modules가 {adapter_info['target_modules']}입니다. linear_table 의 층 이름과 같아야 합니다."

# 저장한 폴더에는 어댑터 파일만 있고 모델 가중치는 없는지
assert "adapter_config.json" in saved_files and "adapter_model.safetensors" in saved_files, f"saved_files가 {saved_files}입니다. lora_model 을 commerce_lora_adapter 폴더에 저장했는지 확인하세요."
assert "model.safetensors" not in saved_files, "어댑터 폴더에 모델 가중치가 있습니다. 어댑터를 붙인 lora_model로 저장했는지 확인하세요."

# 다시 불러온 모델은 새 베이스 위에 학습한 B 값을 그대로 얹고 같은 답을 내는지
assert isinstance(reloaded_model, PeftModel) and reloaded_model is not lora_model, "reloaded_model은 PeftModel.from_pretrained 가 돌려준 새 모델이어야 합니다."
assert reloaded_model.get_base_model() is reload_base and reload_base is not lora_model.get_base_model(), "reloaded_model 의 베이스가 새로 불러온 reload_base 가 아닙니다. 학습한 base_model 은 다시 쓰지 마세요."
reloaded_gate = reloaded_model.get_base_model().model.layers[0].mlp.gate_proj
assert torch.equal(reloaded_gate.lora_B["default"].weight, gate_layer.lora_B["default"].weight), "다시 불러온 어댑터의 B 값이 학습한 값과 다릅니다. commerce_lora_adapter 폴더를 불러왔는지 확인하세요."
assert isinstance(reloaded_answer, str) and reloaded_answer == check_table["LoRA 답변"][0], "다시 불러온 어댑터의 답이 저장 전 LoRA 답변과 다릅니다. check_rows 첫 질문을 system_prompt 기본값으로 물었는지 확인하세요."
print("✅ 통과!")